In [1]:
!pip install underthesea flask pyngrok joblib scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.4/978.4 kB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 55.0 MB/s eta 0:00:00


In [3]:
from joblib import load
from underthesea import word_tokenize
import string
from flask import Flask, request, jsonify
from threading import Thread

def clean_text_vi(text):
    text = text.lower()
    tokens = word_tokenize(text)
    out = []
    for tok in tokens:
        for sub in tok.split():
            if sub not in string.punctuation:
                out.append(sub)
    return " ".join(out)

# Tải Pipeline Model
try:
    pipe = load("model_pipeline.joblib")
    print("Mô hình 'model_pipeline.joblib' đã được tải thành công.")
except Exception as e:
    print(f"LỖI khi tải mô hình: {e}")

# Định nghĩa Flask App
app = Flask(__name__)

@app.route("/predict", methods=["POST"])
def predict():
    if 'pipe' not in globals():
         return jsonify({"error": "Model chưa được tải."}), 500

    data = request.get_json(force=True)
    txt = data.get("text", "")
    pred = pipe.predict([txt])[0]
    return jsonify({"text": txt, "sentiment": pred})

def run_flask_app():
    """Hàm chạy Flask trên port 5000 trong luồng riêng."""
    app.run(port=5000, debug=False, use_reloader=False)

# Khởi chạy Flask trên luồng nền (background thread)
flask_thread = Thread(target=run_flask_app)
flask_thread.start()

print("Flask App đang chạy ngầm trên port 5000...")

Mô hình 'model_pipeline.joblib' đã được tải thành công.
Flask App đang chạy ngầm trên port 5000...
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000


In [6]:
from pyngrok import ngrok
from pyngrok.exception import PyngrokNgrokHTTPError

NGROK_TOKEN = ""
ngrok.set_auth_token(NGROK_TOKEN)

try:
    # Mở tunnel tới cổng 5000 (cổng mà Flask đang chạy)
    # Đóng mọi tunnel cũ nếu có
    ngrok.kill()
    public_url = ngrok.connect(5000)

    print(f"Ngrok Tunnel đã được mở.")
    print(f"Địa chỉ Public URL: {public_url}")

except PyngrokNgrokHTTPError as e:
    print(f"LỖI ngrok: Vui lòng kiểm tra lại Authtoken. Lỗi chi tiết: {e}")
except Exception as e:
    print(f"LỖI không xác định khi mở ngrok: {e}")

Ngrok Tunnel đã được mở.
Địa chỉ Public URL: NgrokTunnel: "https://insensitively-unexistential-bradly.ngrok-free.dev" -> "http://localhost:5000"


In [7]:
import requests
import json
from pyngrok import ngrok

# Lấy địa chỉ URL công khai từ ngrok
try:
    # Lấy tunnel đang hoạt động (cần phải chạy Cell 3 trước đó)
    tunnel = ngrok.get_tunnels()[0]
    public_url_str = tunnel.public_url

    # Dữ liệu gửi đi
    test_data = {"text": "dịch vụ rất tốt, tôi rất hài lòng với chất lượng sản phẩm"}

    # Gửi yêu cầu POST
    resp = requests.post(public_url_str + "/predict", json=test_data)

    print(f"Gửi POST tới: {public_url_str}/predict")
    print(f"Request data: {json.dumps(test_data, ensure_ascii=False)}")
    print("--------------------------------------------------")

    # In kết quả
    resp_json = resp.json()
    print("Kết quả Phản hồi:")
    print(json.dumps(resp_json, indent=4, ensure_ascii=False))

except IndexError:
    print("LỖI: ngrok tunnel chưa hoạt động. Vui lòng kiểm tra lại Cell 3.")
except requests.exceptions.ConnectionError:
    print("LỖI: Không thể kết nối tới Flask App qua ngrok. Đảm bảo Flask App đang chạy ở Cell 2.")
except Exception as e:
    print(f"LỖI không xác định: {e}")

INFO:werkzeug:127.0.0.1 - - [29/Nov/2025 16:59:09] "POST /predict HTTP/1.1" 200 -


Gửi POST tới: https://insensitively-unexistential-bradly.ngrok-free.dev/predict
Request data: {"text": "dịch vụ rất tốt, tôi rất hài lòng với chất lượng sản phẩm"}
--------------------------------------------------
Kết quả Phản hồi:
{
    "sentiment": "2",
    "text": "dịch vụ rất tốt, tôi rất hài lòng với chất lượng sản phẩm"
}
